In [4]:
# Cell 1 - Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_log_error

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

print("Imports loaded successfully.")

Imports loaded successfully.


In [3]:
# Cell 2 - Load raw data

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train.head()

Train shape: (1460, 81)
Test shape : (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
# Cell 3 - Separate features and target

X = train.drop(columns=["SalePrice"]).copy()
y = np.log1p(train["SalePrice"])

X_test = test.copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)

print("\nTarget (log1p SalePrice):")
print(y.describe())

X shape     : (1460, 80)
y shape     : (1460,)
X_test shape: (1459, 80)

Target (log1p SalePrice):
count    1460.000000
mean       12.024057
std         0.399449
min        10.460271
25%        11.775105
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64


In [6]:
# Cell 4 - Leakage-safe feature engineering

def add_features(df):
    data = df.copy()

    # Total living area
    data["TotalSF"] = (
        data["TotalBsmtSF"].fillna(0)
        + data["1stFlrSF"].fillna(0)
        + data["2ndFlrSF"].fillna(0)
    )

    # Total number of bathrooms
    data["TotalBath"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    # Age-related features
    data["HouseAge"] = data["YrSold"] - data["YearBuilt"]
    data["RemodAge"] = data["YrSold"] - data["YearRemodAdd"]

    # Total porch area
    data["TotalPorchSF"] = (
        data["OpenPorchSF"].fillna(0)
        + data["EnclosedPorch"].fillna(0)
        + data["3SsnPorch"].fillna(0)
        + data["ScreenPorch"].fillna(0)
        + data["WoodDeckSF"].fillna(0)
    )

    # Binary features
    data["HasGarage"] = (data["GarageArea"].fillna(0) > 0).astype(int)
    data["HasBsmt"] = (data["TotalBsmtSF"].fillna(0) > 0).astype(int)
    data["HasFireplace"] = (data["Fireplaces"].fillna(0) > 0).astype(int)
    data["HasPool"] = (data["PoolArea"].fillna(0) > 0).astype(int)

    # Quality / size interactions
    data["Qual_TotalSF"] = data["OverallQual"] * data["TotalSF"]
    data["Qual_GrLivArea"] = data["OverallQual"] * data["GrLivArea"]

    return data


X_fe = add_features(X)
X_test_fe = add_features(X_test)

print("Before feature engineering:", X.shape)
print("After feature engineering :", X_fe.shape)
print("Test shape after FE        :", X_test_fe.shape)

new_features = [col for col in X_fe.columns if col not in X.columns]
print("\nNew features:")
print(new_features)


Before feature engineering: (1460, 80)
After feature engineering : (1460, 91)
Test shape after FE        : (1459, 91)

New features:
['TotalSF', 'TotalBath', 'HouseAge', 'RemodAge', 'TotalPorchSF', 'HasGarage', 'HasBsmt', 'HasFireplace', 'HasPool', 'Qual_TotalSF', 'Qual_GrLivArea']


In [7]:
# Cell 5 - Leakage-free preprocessing

# Identify numerical and categorical columns
numeric_features = X_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features  :", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features      :", len(numeric_features) + len(categorical_features))


# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("\nLeakage-free preprocessor created successfully.")
print("IMPORTANT: The preprocessor has NOT been fitted yet.")

Numerical features  : 48
Categorical features: 43
Total features      : 91

Leakage-free preprocessor created successfully.
IMPORTANT: The preprocessor has NOT been fitted yet.


In [8]:
# Cell 6 - Leakage-free 5-fold CV with Ridge

from sklearn.base import clone

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", Ridge(alpha=10.0))
])

oof_ridge = np.zeros(len(X_fe))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe), start=1):

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    # FIT happens ONLY on the training fold
    ridge_pipeline.fit(X_train_fold, y_train_fold)

    # Validation fold is only transformed/predicted
    valid_pred = ridge_pipeline.predict(X_valid_fold)

    oof_ridge[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean((valid_pred - y_valid_fold) ** 2)
    )

    fold_scores.append(fold_rmse)

    print(f"Fold {fold} RMSLE: {fold_rmse:.5f}")


overall_rmsle = np.sqrt(
    np.mean((oof_ridge - y) ** 2)
)

print("\n------------------------------")
print(f"Mean Fold RMSLE : {np.mean(fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(fold_scores):.5f}")
print(f"OOF RMSLE       : {overall_rmsle:.5f}")
print("------------------------------")

Fold 1 RMSLE: 0.13315
Fold 2 RMSLE: 0.12910
Fold 3 RMSLE: 0.22172
Fold 4 RMSLE: 0.12300
Fold 5 RMSLE: 0.10674

------------------------------
Mean Fold RMSLE : 0.14274
Std Fold RMSLE  : 0.04050
OOF RMSLE       : 0.14838
------------------------------


In [9]:
# Cell 7 - OOF error analysis

error_analysis = pd.DataFrame({
    "Id": train["Id"],
    "ActualLogPrice": y,
    "PredictedLogPrice": oof_ridge
})

error_analysis["Residual"] = (
    error_analysis["ActualLogPrice"]
    - error_analysis["PredictedLogPrice"]
)

error_analysis["AbsError"] = np.abs(
    error_analysis["Residual"]
)

# Convert log predictions back to original price scale
error_analysis["ActualPrice"] = np.expm1(
    error_analysis["ActualLogPrice"]
)

error_analysis["PredictedPrice"] = np.expm1(
    error_analysis["PredictedLogPrice"]
)

# Largest OOF errors
worst_errors = error_analysis.nlargest(
    15,
    "AbsError"
)

print("Top 15 largest OOF prediction errors:\n")

display(
    worst_errors[
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "Residual",
            "AbsError"
        ]
    ]
)

Top 15 largest OOF prediction errors:



,Id,ActualPrice,PredictedPrice,Residual,AbsError
1298,1299,160000.0,2.662808e+06,-2.811957,2.811957
523,524,184750.0,1.076036e+06,-1.762031,1.762031
632,633,82500.0,1.762849e+05,-0.759297,0.759297
30,31,40000.0,8.545037e+04,-0.759043,0.759043
1182,1183,745000.0,3.774395e+05,0.679973,0.679973
462,463,62383.0,1.229676e+05,-0.678621,0.678621
1324,1325,147000.0,2.808474e+05,-0.647376,0.647376
495,496,34900.0,6.634935e+04,-0.642434,0.642434
968,969,37900.0,7.109272e+04,-0.629022,0.629022
812,813,55993.0,9.715282e+04,-0.551051,0.551051


In [10]:
# Cell 8 - Inspect the worst OOF cases and their folds

# Assign each observation to its validation fold
fold_assignment = np.zeros(len(X_fe), dtype=int)

for fold, (_, valid_idx) in enumerate(kf.split(X_fe), start=1):
    fold_assignment[valid_idx] = fold

error_analysis["Fold"] = fold_assignment

# Important original features for inspection
inspection_cols = [
    "Id",
    "SalePrice",
    "GrLivArea",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "Neighborhood",
    "LotArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageCars",
    "GarageArea",
    "SaleCondition"
]

inspection = train[inspection_cols].copy()

inspection["Fold"] = fold_assignment
inspection["OOF_PredictedPrice"] = np.expm1(oof_ridge)
inspection["AbsLogError"] = error_analysis["AbsError"]

worst_inspection = inspection.nlargest(
    15,
    "AbsLogError"
)

display(worst_inspection)

,Id,SalePrice,GrLivArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,Neighborhood,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageCars,GarageArea,SaleCondition,Fold,OOF_PredictedPrice,AbsLogError
1298,1299,160000,5642,10,5,2008,2008,Edwards,63887,6110,4692,950,2,1418,Partial,3,2.662808e+06,2.811957
523,524,184750,4676,10,5,2007,2008,Edwards,40094,3138,3138,1538,3,884,Partial,3,1.076036e+06,1.762031
632,633,82500,1411,7,5,1977,1977,NWAmes,11900,1386,1411,0,2,544,Family,5,1.762849e+05,0.759297
30,31,40000,1317,4,4,1920,1950,IDOTRR,8500,649,649,668,1,250,Normal,1,8.545037e+04,0.759043
1182,1183,745000,4476,10,5,1996,1996,NoRidge,15623,2396,2411,2065,3,813,Abnorml,2,3.774395e+05,0.679973
462,463,62383,864,5,5,1965,1965,Sawyer,8281,864,864,0,1,360,Normal,2,1.229676e+05,0.678621
1324,1325,147000,1795,8,5,2006,2007,Somerst,9986,1795,1795,0,3,895,Partial,3,2.808474e+05,0.647376
495,496,34900,720,4,5,1920,1950,IDOTRR,7879,720,720,0,0,0,Abnorml,4,6.634935e+04,0.642434
968,969,37900,968,3,6,1910,1950,OldTown,5925,600,600,368,0,0,Abnorml,4,7.109272e+04,0.629022
812,813,55993,1044,5,5,1952,1952,IDOTRR,8712,540,1044,0,2,504,Alloca,1,9.715282e+04,0.551051


In [11]:
# Cell 9 - Quantify the influence of the two extreme observations

extreme_ids = [524, 1299]

extreme_mask = train["Id"].isin(extreme_ids)
normal_mask = ~extreme_mask

# OOF RMSLE on all observations
rmsle_all = np.sqrt(
    np.mean((oof_ridge - y) ** 2)
)

# Same OOF predictions, but evaluate without the two extreme observations
rmsle_without_extremes = np.sqrt(
    np.mean(
        (oof_ridge[normal_mask] - y[normal_mask]) ** 2
    )
)

print(f"OOF RMSLE - all observations       : {rmsle_all:.5f}")
print(f"OOF RMSLE - excluding Id 524/1299 : {rmsle_without_extremes:.5f}")

print(
    f"Difference                        : "
    f"{rmsle_all - rmsle_without_extremes:.5f}"
)

print("\nExtreme observations:")
display(
    error_analysis.loc[
        extreme_mask,
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "Fold",
            "AbsError"
        ]
    ]
)

OOF RMSLE - all observations       : 0.14838
OOF RMSLE - excluding Id 524/1299 : 0.12039
Difference                        : 0.02799

Extreme observations:


,Id,ActualPrice,PredictedPrice,Fold,AbsError
523,524,184750.0,1.076036e+06,3,1.762031
1298,1299,160000.0,2.662808e+06,3,2.811957


In [12]:
# Cell 10 - Investigate the two extreme observations

key_features = [
    "GrLivArea",
    "LotArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageArea",
    "OverallQual",
    "SalePrice"
]

print("Dataset percentiles:")
display(
    train[key_features].quantile(
        [0.50, 0.90, 0.95, 0.99, 1.00]
    )
)

print("\nExtreme observations:")
display(
    train.loc[
        train["Id"].isin([524, 1299]),
        [
            "Id",
            "Neighborhood",
            "GrLivArea",
            "LotArea",
            "TotalBsmtSF",
            "1stFlrSF",
            "2ndFlrSF",
            "GarageArea",
            "OverallQual",
            "OverallCond",
            "SaleType",
            "SaleCondition",
            "SalePrice"
        ]
    ]
)

Dataset percentiles:


,GrLivArea,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageArea,OverallQual,SalePrice
0.50,1464.00,9478.50,991.50,1087.00,0.00,480.00,6.0,163000.00
0.90,2158.30,14381.70,1602.20,1680.00,954.20,757.10,8.0,278000.00
0.95,2466.10,17401.15,1753.00,1831.25,1141.05,850.10,8.0,326100.00
0.99,3123.48,37567.64,2155.05,2219.46,1418.92,1002.79,10.0,442567.01
1.00,5642.00,215245.00,6110.00,4692.00,2065.00,1418.00,10.0,755000.00



Extreme observations:


,Id,Neighborhood,GrLivArea,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageArea,OverallQual,OverallCond,SaleType,SaleCondition,SalePrice
523,524,Edwards,4676,40094,3138,3138,1538,884,10,5,New,Partial,184750
1298,1299,Edwards,5642,63887,6110,4692,950,1418,10,5,New,Partial,160000


In [13]:
# Cell 11 - Define a transparent structural outlier rule

# Large houses with unusually low sale prices
outlier_rule = (
    (train["GrLivArea"] > 4000)
    & (train["SalePrice"] < 300000)
)

structural_outliers = train.loc[
    outlier_rule,
    [
        "Id",
        "GrLivArea",
        "OverallQual",
        "Neighborhood",
        "SaleType",
        "SaleCondition",
        "SalePrice"
    ]
]

print("Number of structural outliers:", outlier_rule.sum())
print()

display(structural_outliers)

Number of structural outliers: 2



,Id,GrLivArea,OverallQual,Neighborhood,SaleType,SaleCondition,SalePrice
523,524,4676,10,Edwards,New,Partial,184750
1298,1299,5642,10,Edwards,New,Partial,160000


In [14]:
# Cell 12 - Retrain leakage-free Ridge after structural outlier removal

# Keep observations that do NOT satisfy the structural outlier rule
clean_mask = ~outlier_rule

X_clean = X_fe.loc[clean_mask].reset_index(drop=True)
y_clean = y.loc[clean_mask].reset_index(drop=True)

print("Original training observations:", len(X_fe))
print("Removed structural outliers   :", outlier_rule.sum())
print("Clean training observations   :", len(X_clean))

# New CV from scratch
kf_clean = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_ridge_clean = np.zeros(len(X_clean))
clean_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):

    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # IMPORTANT:
    # Create a fresh pipeline for every fold
    fold_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=10.0))
    ])

    # Preprocessing is fitted ONLY on this training fold
    fold_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = fold_pipeline.predict(
        X_valid_fold
    )

    oof_ridge_clean[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    clean_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


clean_oof_rmsle = np.sqrt(
    np.mean(
        (oof_ridge_clean - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(
    f"Mean Fold RMSLE : "
    f"{np.mean(clean_fold_scores):.5f}"
)
print(
    f"Std Fold RMSLE  : "
    f"{np.std(clean_fold_scores):.5f}"
)
print(
    f"OOF RMSLE       : "
    f"{clean_oof_rmsle:.5f}"
)
print("----------------------------------")

Original training observations: 1460
Removed structural outliers   : 2
Clean training observations   : 1458
Fold 1 RMSLE: 0.12305
Fold 2 RMSLE: 0.11048
Fold 3 RMSLE: 0.11705
Fold 4 RMSLE: 0.12355
Fold 5 RMSLE: 0.10185

----------------------------------
Mean Fold RMSLE : 0.11520
Std Fold RMSLE  : 0.00819
OOF RMSLE       : 0.11549
----------------------------------


In [15]:
# Cell 13 - Experiment summary

experiment_results = pd.DataFrame({
    "Experiment": [
        "Ridge - all observations",
        "Ridge - diagnostic exclusion only",
        "Ridge - retrained after structural outlier rule"
    ],
    "OOF_RMSLE": [
        0.14838,
        0.12039,
        clean_oof_rmsle
    ],
    "Interpretation": [
        "Leakage-free baseline; strongly affected by two extreme observations",
        "Diagnostic only - observations excluded from scoring, no retraining",
        "Full leakage-free CV retrained on 1458 observations"
    ]
})

display(experiment_results)

,Experiment,OOF_RMSLE,Interpretation
0,Ridge - all observations,0.148380,Leakage-free baseline; strongly affected by tw...
1,Ridge - diagnostic exclusion only,0.120390,Diagnostic only - observations excluded from s...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...


In [16]:
# Cell 14 - Leakage-free 5-fold CV with XGBoost

xgb_oof = np.zeros(len(X_clean))
xgb_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # Fresh pipeline for every fold
    xgb_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),

        ("model", xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Preprocessing + model are fitted ONLY on training fold
    xgb_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = xgb_pipeline.predict(
        X_valid_fold
    )

    xgb_oof[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    xgb_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


xgb_oof_rmsle = np.sqrt(
    np.mean(
        (xgb_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(
    f"Mean Fold RMSLE : "
    f"{np.mean(xgb_fold_scores):.5f}"
)
print(
    f"Std Fold RMSLE  : "
    f"{np.std(xgb_fold_scores):.5f}"
)
print(
    f"OOF RMSLE       : "
    f"{xgb_oof_rmsle:.5f}"
)
print("----------------------------------")

Fold 1 RMSLE: 0.11827
Fold 2 RMSLE: 0.10966
Fold 3 RMSLE: 0.12039
Fold 4 RMSLE: 0.12158
Fold 5 RMSLE: 0.10612

----------------------------------
Mean Fold RMSLE : 0.11520
Std Fold RMSLE  : 0.00617
OOF RMSLE       : 0.11537
----------------------------------


In [17]:
# Cell 15 - Update experiment summary with XGBoost

new_result = pd.DataFrame({
    "Experiment": [
        "XGBoost - leakage-free CV after structural outlier rule"
    ],
    "OOF_RMSLE": [
        xgb_oof_rmsle
    ],
    "Interpretation": [
        "Slightly better OOF RMSLE than Ridge and lower fold-to-fold variance"
    ]
})

experiment_results = pd.concat(
    [experiment_results, new_result],
    ignore_index=True
)

display(experiment_results)

,Experiment,OOF_RMSLE,Interpretation
0,Ridge - all observations,0.148380,Leakage-free baseline; strongly affected by tw...
1,Ridge - diagnostic exclusion only,0.120390,Diagnostic only - observations excluded from s...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
3,XGBoost - leakage-free CV after structural out...,0.115370,Slightly better OOF RMSLE than Ridge and lower...


In [18]:
# Cell 16 - Compare Ridge and XGBoost OOF predictions

ridge_errors = y_clean - oof_ridge_clean
xgb_errors = y_clean - xgb_oof

error_correlation = np.corrcoef(
    ridge_errors,
    xgb_errors
)[0, 1]

prediction_correlation = np.corrcoef(
    oof_ridge_clean,
    xgb_oof
)[0, 1]

print(
    f"Prediction correlation : "
    f"{prediction_correlation:.5f}"
)

print(
    f"Error correlation      : "
    f"{error_correlation:.5f}"
)

print("\nIndividual OOF RMSLE:")
print(f"Ridge   : {clean_oof_rmsle:.5f}")
print(f"XGBoost : {xgb_oof_rmsle:.5f}")

Prediction correlation : 0.98551
Error correlation      : 0.83920

Individual OOF RMSLE:
Ridge   : 0.11549
XGBoost : 0.11537


In [19]:
# Cell 17 - Ridge + XGBoost OOF blend

blend_results = []

# Test several Ridge/XGBoost weights
for ridge_weight in np.arange(0.0, 1.01, 0.1):

    xgb_weight = 1.0 - ridge_weight

    blend_oof = (
        ridge_weight * oof_ridge_clean
        + xgb_weight * xgb_oof
    )

    blend_rmsle = np.sqrt(
        np.mean(
            (blend_oof - y_clean) ** 2
        )
    )

    blend_results.append({
        "Ridge_weight": ridge_weight,
        "XGBoost_weight": xgb_weight,
        "OOF_RMSLE": blend_rmsle
    })


blend_results = pd.DataFrame(blend_results)

display(
    blend_results.sort_values("OOF_RMSLE")
)

best_blend = blend_results.loc[
    blend_results["OOF_RMSLE"].idxmin()
]

print("\nBest OOF blend:")
print(
    f"Ridge weight   : "
    f"{best_blend['Ridge_weight']:.1f}"
)
print(
    f"XGBoost weight : "
    f"{best_blend['XGBoost_weight']:.1f}"
)
print(
    f"OOF RMSLE      : "
    f"{best_blend['OOF_RMSLE']:.5f}"
)

,Ridge_weight,XGBoost_weight,OOF_RMSLE
5,0.5,0.5,0.110691
4,0.4,0.6,0.110872
6,0.6,0.4,0.110897
3,0.3,0.7,0.111438
7,0.7,0.3,0.111488
2,0.2,0.8,0.112383
8,0.8,0.2,0.112457
1,0.1,0.9,0.113698
9,0.9,0.1,0.113795
0,0.0,1.0,0.115370



Best OOF blend:
Ridge weight   : 0.5
XGBoost weight : 0.5
OOF RMSLE      : 0.11069


In [20]:
# Cell 18 - Fold-by-fold evaluation of the 50/50 OOF blend

best_blend_oof = (
    0.5 * oof_ridge_clean
    + 0.5 * xgb_oof
)

blend_fold_scores = []

for fold, (_, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    fold_score = np.sqrt(
        np.mean(
            (
                best_blend_oof[valid_idx]
                - y_clean.iloc[valid_idx]
            ) ** 2
        )
    )

    blend_fold_scores.append(fold_score)

    print(
        f"Fold {fold} Blend RMSLE: "
        f"{fold_score:.5f}"
    )


blend_oof_rmsle = np.sqrt(
    np.mean(
        (best_blend_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(f"Mean Fold RMSLE : {np.mean(blend_fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(blend_fold_scores):.5f}")
print(f"OOF RMSLE       : {blend_oof_rmsle:.5f}")
print("----------------------------------")

Fold 1 Blend RMSLE: 0.11664
Fold 2 Blend RMSLE: 0.10544
Fold 3 Blend RMSLE: 0.11515
Fold 4 Blend RMSLE: 0.11577
Fold 5 Blend RMSLE: 0.09937

----------------------------------
Mean Fold RMSLE : 0.11047
Std Fold RMSLE  : 0.00688
OOF RMSLE       : 0.11069
----------------------------------


In [21]:
# Cell 19 - Add the best blend to the experiment summary

blend_result = pd.DataFrame({
    "Experiment": [
        "50/50 Ridge + XGBoost OOF blend"
    ],
    "OOF_RMSLE": [
        blend_oof_rmsle
    ],
    "Interpretation": [
        "Best result so far; stable improvement from complementary OOF errors"
    ]
})

experiment_results = pd.concat(
    [experiment_results, blend_result],
    ignore_index=True
)

display(
    experiment_results.sort_values("OOF_RMSLE")
)

,Experiment,OOF_RMSLE,Interpretation
4,50/50 Ridge + XGBoost OOF blend,0.110691,Best result so far; stable improvement from co...
3,XGBoost - leakage-free CV after structural out...,0.115370,Slightly better OOF RMSLE than Ridge and lower...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
1,Ridge - diagnostic exclusion only,0.120390,Diagnostic only - observations excluded from s...
0,Ridge - all observations,0.148380,Leakage-free baseline; strongly affected by tw...


In [22]:
# Cell 20 - Leakage-free 5-fold CV with LightGBM

lgb_oof = np.zeros(len(X_clean))
lgb_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # Fresh pipeline for every fold
    lgb_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),

        ("model", lgb.LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ))
    ])

    # Preprocessing and model fit ONLY on training fold
    lgb_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = lgb_pipeline.predict(
        X_valid_fold
    )

    lgb_oof[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    lgb_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


lgb_oof_rmsle = np.sqrt(
    np.mean(
        (lgb_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(f"Mean Fold RMSLE : {np.mean(lgb_fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(lgb_fold_scores):.5f}")
print(f"OOF RMSLE       : {lgb_oof_rmsle:.5f}")
print("----------------------------------")

Fold 1 RMSLE: 0.13272
Fold 2 RMSLE: 0.11759
Fold 3 RMSLE: 0.12879
Fold 4 RMSLE: 0.12758
Fold 5 RMSLE: 0.10992

----------------------------------
Mean Fold RMSLE : 0.12332
Std Fold RMSLE  : 0.00835
OOF RMSLE       : 0.12361
----------------------------------


In [23]:
# Cell 21 - Compare OOF error correlations

ridge_errors = y_clean - oof_ridge_clean
xgb_errors = y_clean - xgb_oof
lgb_errors = y_clean - lgb_oof

error_corr = pd.DataFrame({
    "Ridge": ridge_errors,
    "XGBoost": xgb_errors,
    "LightGBM": lgb_errors
}).corr()

print("OOF error correlation:")
display(error_corr)

print("\nIndividual OOF RMSLE:")
print(f"Ridge    : {clean_oof_rmsle:.5f}")
print(f"XGBoost  : {xgb_oof_rmsle:.5f}")
print(f"LightGBM : {lgb_oof_rmsle:.5f}")
print(f"Best 50/50 blend: {blend_oof_rmsle:.5f}")

OOF error correlation:


,Ridge,XGBoost,LightGBM
Ridge,1.000000,0.839196,0.785819
XGBoost,0.839196,1.000000,0.921817
LightGBM,0.785819,0.921817,1.000000



Individual OOF RMSLE:
Ridge    : 0.11549
XGBoost  : 0.11537
LightGBM : 0.12361
Best 50/50 blend: 0.11069


In [24]:
# Cell 22 - Controlled three-model OOF blend test

three_model_results = []

# Test small LightGBM contributions only.
# Ridge and XGBoost share the remaining weight equally.
for lgb_weight in [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:

    remaining_weight = 1.0 - lgb_weight

    ridge_weight = remaining_weight / 2
    xgb_weight = remaining_weight / 2

    blend_3_oof = (
        ridge_weight * oof_ridge_clean
        + xgb_weight * xgb_oof
        + lgb_weight * lgb_oof
    )

    rmsle = np.sqrt(
        np.mean(
            (blend_3_oof - y_clean) ** 2
        )
    )

    three_model_results.append({
        "Ridge_weight": ridge_weight,
        "XGBoost_weight": xgb_weight,
        "LightGBM_weight": lgb_weight,
        "OOF_RMSLE": rmsle
    })


three_model_results = pd.DataFrame(
    three_model_results
)

display(
    three_model_results.sort_values("OOF_RMSLE")
)

best_three_model = three_model_results.loc[
    three_model_results["OOF_RMSLE"].idxmin()
]

print("\nBest three-model test:")
print(
    f"Ridge    : "
    f"{best_three_model['Ridge_weight']:.2f}"
)
print(
    f"XGBoost  : "
    f"{best_three_model['XGBoost_weight']:.2f}"
)
print(
    f"LightGBM : "
    f"{best_three_model['LightGBM_weight']:.2f}"
)
print(
    f"OOF RMSLE: "
    f"{best_three_model['OOF_RMSLE']:.5f}"
)

print(
    f"\nCurrent 50/50 Ridge-XGBoost: "
    f"{blend_oof_rmsle:.5f}"
)

,Ridge_weight,XGBoost_weight,LightGBM_weight,OOF_RMSLE
0,0.500,0.500,0.00,0.110691
1,0.475,0.475,0.05,0.110695
2,0.450,0.450,0.10,0.110770
3,0.425,0.425,0.15,0.110917
4,0.400,0.400,0.20,0.111135
5,0.375,0.375,0.25,0.111423
6,0.350,0.350,0.30,0.111782



Best three-model test:
Ridge    : 0.50
XGBoost  : 0.50
LightGBM : 0.00
OOF RMSLE: 0.11069

Current 50/50 Ridge-XGBoost: 0.11069


In [25]:
# Cell 23 - Record the LightGBM experiment and model-selection decision

lgb_result = pd.DataFrame({
    "Experiment": [
        "LightGBM - leakage-free CV after structural outlier rule",
        "Best Ridge + XGBoost + LightGBM controlled blend"
    ],
    "OOF_RMSLE": [
        lgb_oof_rmsle,
        best_three_model["OOF_RMSLE"]
    ],
    "Interpretation": [
        "Weaker standalone model; high error correlation with XGBoost",
        "Optimal LightGBM weight was 0%; no improvement over the 50/50 Ridge-XGBoost blend"
    ]
})

experiment_results = pd.concat(
    [experiment_results, lgb_result],
    ignore_index=True
)

display(
    experiment_results.sort_values("OOF_RMSLE")
)

print("\nMODEL SELECTION DECISION")
print("------------------------")
print("Selected final ensemble: 50% Ridge + 50% XGBoost")
print(f"Best OOF RMSLE          : {blend_oof_rmsle:.5f}")
print("LightGBM                : excluded")
print("Reason                  : no OOF improvement and unnecessary complexity")

,Experiment,OOF_RMSLE,Interpretation
4,50/50 Ridge + XGBoost OOF blend,0.110691,Best result so far; stable improvement from co...
6,Best Ridge + XGBoost + LightGBM controlled blend,0.110691,Optimal LightGBM weight was 0%; no improvement...
3,XGBoost - leakage-free CV after structural out...,0.115370,Slightly better OOF RMSLE than Ridge and lower...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
1,Ridge - diagnostic exclusion only,0.120390,Diagnostic only - observations excluded from s...
5,LightGBM - leakage-free CV after structural ou...,0.123607,Weaker standalone model; high error correlatio...
0,Ridge - all observations,0.148380,Leakage-free baseline; strongly affected by tw...



MODEL SELECTION DECISION
------------------------
Selected final ensemble: 50% Ridge + 50% XGBoost
Best OOF RMSLE          : 0.11069
LightGBM                : excluded
Reason                  : no OOF improvement and unnecessary complexity


In [26]:
# Cell 24 - Final ensemble OOF error analysis

final_error_analysis = pd.DataFrame({
    "Id": train.loc[clean_mask, "Id"].reset_index(drop=True),
    "ActualLogPrice": y_clean,
    "PredictedLogPrice": best_blend_oof
})

# Residuals in log space
final_error_analysis["Residual"] = (
    final_error_analysis["ActualLogPrice"]
    - final_error_analysis["PredictedLogPrice"]
)

final_error_analysis["AbsLogError"] = np.abs(
    final_error_analysis["Residual"]
)

# Convert predictions back to original dollar scale
final_error_analysis["ActualPrice"] = np.expm1(
    final_error_analysis["ActualLogPrice"]
)

final_error_analysis["PredictedPrice"] = np.expm1(
    final_error_analysis["PredictedLogPrice"]
)

final_error_analysis["PriceError"] = (
    final_error_analysis["PredictedPrice"]
    - final_error_analysis["ActualPrice"]
)

# Percentage error
final_error_analysis["PercentError"] = (
    final_error_analysis["PriceError"]
    / final_error_analysis["ActualPrice"]
) * 100


print("FINAL ENSEMBLE ERROR SUMMARY")
print("----------------------------")

print(
    f"Mean residual      : "
    f"{final_error_analysis['Residual'].mean():.5f}"
)

print(
    f"Median residual    : "
    f"{final_error_analysis['Residual'].median():.5f}"
)

print(
    f"Mean absolute log error: "
    f"{final_error_analysis['AbsLogError'].mean():.5f}"
)

print(
    f"Median absolute log error: "
    f"{final_error_analysis['AbsLogError'].median():.5f}"
)


print("\nTop 15 largest final ensemble OOF errors:")

display(
    final_error_analysis.nlargest(
        15,
        "AbsLogError"
    )[
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "PriceError",
            "PercentError",
            "Residual",
            "AbsLogError"
        ]
    ]
)

FINAL ENSEMBLE ERROR SUMMARY
----------------------------
Mean residual      : 0.00071
Median residual    : 0.00395
Mean absolute log error: 0.07482
Median absolute log error: 0.05283

Top 15 largest final ensemble OOF errors:


,Id,ActualPrice,PredictedPrice,PriceError,PercentError,Residual,AbsLogError
631,633,82500.0,178869.330998,96369.330998,116.811310,-0.773851,0.773851
30,31,40000.0,85646.797354,45646.797354,114.116993,-0.761339,0.761339
462,463,62383.0,122505.193938,60122.193938,96.375926,-0.674853,0.674853
495,496,34900.0,67419.070329,32519.070329,93.177852,-0.658427,0.658427
1322,1325,147000.0,283252.704087,136252.704087,92.688914,-0.655904,0.655904
88,89,85000.0,48777.961720,-36222.038280,-42.614163,0.555364,0.555364
969,971,135000.0,77732.526500,-57267.473500,-42.420351,0.551996,0.551996
915,917,35311.0,60609.425800,25298.425800,71.644603,-0.540244,0.540244
967,969,37900.0,64144.557552,26244.557552,69.246854,-0.526177,0.526177
1430,1433,64500.0,105749.490523,41249.490523,63.952698,-0.494402,0.494402


In [28]:
# Cell 25 - Error analysis by price range

final_error_analysis["PriceRange"] = pd.qcut(
    final_error_analysis["ActualPrice"],
    q=4,
    labels=[
        "Low",
        "Lower-Middle",
        "Upper-Middle",
        "High"
    ]
)

price_range_analysis = (
    final_error_analysis
    .groupby("PriceRange", observed=True)
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

display(price_range_analysis)
print("\nRMSLE by price range:")

for price_range in final_error_analysis["PriceRange"].cat.categories:

    group = final_error_analysis[
        final_error_analysis["PriceRange"] == price_range
    ]

    group_rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    print(
        f"{price_range:15s}: "
        f"{group_rmsle:.5f}"
    )


,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError
PriceRange,,,,,,
Low,365,105831.594521,109357.665520,0.099270,0.068678,4.924885
Lower-Middle,366,145001.016393,144277.763372,0.067121,0.047107,-0.458871
Upper-Middle,365,185263.052055,185162.433216,0.058009,0.044832,-0.052943
High,362,288619.552486,280922.851814,0.074919,0.058435,-2.134286



RMSLE by price range:
Low            : 0.15106
Lower-Middle   : 0.09920
Upper-Middle   : 0.07768
High           : 0.10152


In [30]:
# Cell 26 - Error analysis by OverallQual

# Add OverallQual from the clean training data
clean_train = train.loc[clean_mask].reset_index(drop=True)

final_error_analysis["OverallQual"] = clean_train["OverallQual"]

quality_analysis = (
    final_error_analysis
    .groupby("OverallQual")
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

display(quality_analysis)


print("\nRMSLE by OverallQual:")

for quality in sorted(
    final_error_analysis["OverallQual"].unique()
):
    group = final_error_analysis[
        final_error_analysis["OverallQual"] == quality
    ]

    group_rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    print(
        f"OverallQual {quality:2d} "
        f"(n={len(group):3d}): "
        f"{group_rmsle:.5f}"
    )
    print("\nRMSLE by OverallQual:")

for quality in sorted(
    final_error_analysis["OverallQual"].unique()
):
    group = final_error_analysis[
        final_error_analysis["OverallQual"] == quality
    ]

    group_rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    print(
        f"OverallQual {quality:2d} "
        f"(n={len(group):3d}): "
        f"{group_rmsle:.5f}"
    )

,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError
OverallQual,,,,,,
1,2,50150.000000,55658.543934,0.232082,0.232082,16.743031
2,3,51770.333333,57104.174686,0.251799,0.193990,18.716589
3,20,87473.750000,84622.010204,0.152210,0.078304,0.786349
4,116,108420.655172,105906.788697,0.117941,0.087588,0.766617
5,397,133523.347607,132251.180567,0.074970,0.051726,0.194974
6,374,161603.034759,161386.796448,0.063130,0.049731,0.631529
7,319,207716.423197,207468.456579,0.066092,0.048265,0.829491
8,168,274735.535714,274393.005316,0.071623,0.047949,1.064773
9,43,367513.023256,353802.472581,0.073524,0.044426,-2.589547



RMSLE by OverallQual:
OverallQual  1 (n=  2): 0.26509

RMSLE by OverallQual:
OverallQual  2 (n=  3): 0.33163

RMSLE by OverallQual:
OverallQual  3 (n= 20): 0.22312

RMSLE by OverallQual:
OverallQual  4 (n=116): 0.17017

RMSLE by OverallQual:
OverallQual  5 (n=397): 0.10936

RMSLE by OverallQual:
OverallQual  6 (n=374): 0.08554

RMSLE by OverallQual:
OverallQual  7 (n=319): 0.09442

RMSLE by OverallQual:
OverallQual  8 (n=168): 0.11123

RMSLE by OverallQual:
OverallQual  9 (n= 43): 0.09902

RMSLE by OverallQual:
OverallQual 10 (n= 16): 0.11454

RMSLE by OverallQual:
OverallQual  1 (n=  2): 0.26509
OverallQual  2 (n=  3): 0.33163
OverallQual  3 (n= 20): 0.22312
OverallQual  4 (n=116): 0.17017
OverallQual  5 (n=397): 0.10936
OverallQual  6 (n=374): 0.08554
OverallQual  7 (n=319): 0.09442
OverallQual  8 (n=168): 0.11123
OverallQual  9 (n= 43): 0.09902
OverallQual 10 (n= 16): 0.11454


In [31]:
# Cell 27 - Error analysis by Neighborhood

final_error_analysis["Neighborhood"] = clean_train["Neighborhood"]

neighborhood_analysis = (
    final_error_analysis
    .groupby("Neighborhood")
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

# Calculate RMSLE for each neighborhood
neighborhood_rmsle = {}

for neighborhood in final_error_analysis["Neighborhood"].unique():

    group = final_error_analysis[
        final_error_analysis["Neighborhood"] == neighborhood
    ]

    rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    neighborhood_rmsle[neighborhood] = rmsle


neighborhood_analysis["RMSLE"] = pd.Series(
    neighborhood_rmsle
)

# Sort from worst to best RMSLE
neighborhood_analysis = neighborhood_analysis.sort_values(
    "RMSLE",
    ascending=False
)

display(neighborhood_analysis)

,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError,RMSLE
Neighborhood,,,,,,,
IDOTRR,37,100123.783784,98921.296607,0.164743,0.116845,5.848005,0.245407
ClearCr,28,212565.428571,206366.882200,0.111855,0.097574,-1.591176,0.149295
OldTown,113,128225.300885,126989.602558,0.106314,0.075916,1.705186,0.145665
SWISU,25,142591.360000,142261.218393,0.099930,0.066666,0.330729,0.130927
StoneBr,25,310499.000000,294969.970356,0.084123,0.062317,-3.625817,0.128819
Edwards,98,127318.571429,126089.203236,0.094585,0.072980,0.940108,0.127171
BrkSide,58,124834.051724,121445.550842,0.093167,0.075616,-0.087614,0.125506
Sawyer,74,136793.135135,136900.970026,0.072854,0.050255,1.346261,0.120099
NWAmes,73,189050.068493,190843.515897,0.068968,0.047860,1.640463,0.118259


In [32]:
# Cell 28 - Final training on all clean training data

# Final Ridge pipeline
final_ridge = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", Ridge(alpha=10.0))
])

# Final XGBoost pipeline
final_xgb = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training final Ridge...")
final_ridge.fit(X_clean, y_clean)

print("Training final XGBoost...")
final_xgb.fit(X_clean, y_clean)

print("\nFinal models trained successfully.")
print("Training observations:", len(X_clean))
print("Test observations    :", len(X_test_fe))
print("Final ensemble       : 50% Ridge + 50% XGBoost")

Training final Ridge...
Training final XGBoost...

Final models trained successfully.
Training observations: 1458
Test observations    : 1459
Final ensemble       : 50% Ridge + 50% XGBoost


In [33]:
# Cell 29 - Final test predictions and Kaggle submission

# Predict log(SalePrice) with both final models
ridge_test_pred_log = final_ridge.predict(X_test_fe)
xgb_test_pred_log = final_xgb.predict(X_test_fe)

# Final 50/50 ensemble in log space
final_test_pred_log = (
    0.5 * ridge_test_pred_log
    + 0.5 * xgb_test_pred_log
)

# Convert back to original SalePrice scale
final_test_pred = np.expm1(final_test_pred_log)

# Safety checks
print("Prediction checks")
print("-----------------")
print("Number of predictions :", len(final_test_pred))
print("NaN predictions       :", np.isnan(final_test_pred).sum())
print("Infinite predictions  :", np.isinf(final_test_pred).sum())
print("Negative predictions  :", (final_test_pred < 0).sum())

print("\nPrediction summary:")
print(pd.Series(final_test_pred).describe())

# Create Kaggle submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_test_pred
})

# Final validation
assert len(submission) == len(test)
assert submission["SalePrice"].notna().all()
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

# Save to submissions folder
submission_path = (
    "submissions/"
    "submission_leakage_free_ridge_xgb_50_50.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("\nSubmission created successfully:")
print(submission_path)

display(submission.head(10))

Prediction checks
-----------------
Number of predictions : 1459
NaN predictions       : 0
Infinite predictions  : 0
Negative predictions  : 0

Prediction summary:
count    1.459000e+03
mean     1.765932e+05
std      7.970477e+04
min      4.276149e+04
25%      1.256362e+05
50%      1.551604e+05
75%      2.075400e+05
max      1.029621e+06
dtype: float64

Submission created successfully:
submissions/submission_leakage_free_ridge_xgb_50_50.csv


,Id,SalePrice
0,1461,119833.139821
1,1462,156327.254481
2,1463,178708.208155
3,1464,197252.972905
4,1465,186349.904752
5,1466,171382.778042
6,1467,177202.983837
7,1468,164030.612982
8,1469,187212.482609
9,1470,120520.029010


In [35]:
# Cell 30 - Final prediction sanity check

prediction_check = X_test_fe.copy()

prediction_check["Id"] = test["Id"].values
prediction_check["PredictedSalePrice"] = final_test_pred

check_columns = [
    "Id",
    "Neighborhood",
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "YearBuilt",
    "GarageCars",
    "PredictedSalePrice"
]

print("10 highest predicted prices:")

display(
    prediction_check
    .nlargest(10, "PredictedSalePrice")[check_columns]
)

print("\n10 lowest predicted prices:")

display(
    prediction_check
    .nsmallest(10, "PredictedSalePrice")[check_columns]
)

10 highest predicted prices:


,Id,Neighborhood,OverallQual,GrLivArea,TotalSF,YearBuilt,GarageCars,PredictedSalePrice
1089,2550,Edwards,10,5095,10190.0,2008,3.0,1.029621e+06
1222,2683,NoRidge,9,3500,5233.0,1993,3.0,5.749555e+05
203,1664,NridgHt,10,2674,5304.0,2007,3.0,5.287747e+05
19,1480,NridgHt,9,2696,5542.0,2003,3.0,5.280881e+05
803,2264,StoneBr,9,2338,4998.0,2006,3.0,5.169918e+05
1168,2629,StoneBr,10,3390,4918.0,2006,3.0,5.125362e+05
834,2295,NridgHt,10,2290,4610.0,2007,3.0,5.035883e+05
1170,2631,StoneBr,10,2698,4548.0,2006,3.0,5.007637e+05
832,2293,NridgHt,9,2552,5104.0,2007,3.0,4.994166e+05
217,1678,NridgHt,10,2492,4984.0,2004,3.0,4.961079e+05



10 lowest predicted prices:


,Id,Neighborhood,OverallQual,GrLivArea,TotalSF,YearBuilt,GarageCars,PredictedSalePrice
362,1823,IDOTRR,3,797,1042.0,1900,0.0,42761.486317
1431,2892,IDOTRR,3,729,729.0,1945,0.0,43690.760606
1433,2894,IDOTRR,3,936,1152.0,1916,0.0,47533.414975
1331,2792,IDOTRR,3,1020,2040.0,1918,0.0,53640.298291
453,1914,IDOTRR,4,572,1144.0,1925,1.0,54960.625447
1411,2872,Edwards,2,498,996.0,1922,1.0,56055.623385
387,1848,Sawyer,2,660,660.0,1947,0.0,56279.113387
76,1537,OldTown,2,832,1510.0,1923,2.0,56659.650157
354,1815,OldTown,2,612,612.0,1940,1.0,56829.485764
1428,2889,IDOTRR,4,672,1104.0,1925,0.0,57011.100007


In [36]:
# Cell 31 - OOF vs Kaggle generalization gap

oof_score = blend_oof_rmsle
kaggle_public_score = 0.12808

generalization_gap = kaggle_public_score - oof_score

print("FINAL GENERALIZATION CHECK")
print("--------------------------")
print(f"OOF RMSLE          : {oof_score:.5f}")
print(f"Kaggle Public Score: {kaggle_public_score:.5f}")
print(f"Generalization Gap : {generalization_gap:.5f}")

FINAL GENERALIZATION CHECK
--------------------------
OOF RMSLE          : 0.11069
Kaggle Public Score: 0.12808
Generalization Gap : 0.01739


## Generalization Analysis

The final selected model is a 50/50 blend of Ridge Regression and XGBoost.

### Validation Results

- Leakage-free OOF RMSLE: **0.11069**
- Kaggle Public Score: **0.12808**
- Generalization gap: **0.01739**

The Kaggle score is worse than the cross-validation estimate, which indicates that the
cross-validation performance is somewhat optimistic relative to unseen competition data.

This difference should not be hidden or treated simply as a reason to continue tuning
against the public leaderboard. Repeatedly modifying the model based on the Kaggle score
would effectively turn the public leaderboard into an additional validation set and could
lead to leaderboard overfitting.

Several factors may contribute to the observed gap:

1. **Sampling variation** — the Kaggle public leaderboard evaluates only part of the hidden
   test targets, while cross-validation evaluates folds drawn from the available training data.

2. **Rare and extreme properties** — the error analysis showed that model performance is not
   uniform across the housing market. Low-price properties, low OverallQual groups, and some
   neighborhoods are substantially more difficult to predict.

3. **Model limitations** — Ridge and XGBoost reduce each other's OOF errors when blended, but
   neither model can fully capture unusual transactions or properties whose sale prices differ
   substantially from what their structural characteristics suggest.

### Practical Interpretation

The model should therefore not be described only by its overall RMSLE.

The segment-level analysis showed that prediction reliability varies considerably across
different types of properties. For example, the low-price segment had an RMSLE of **0.15106**,
while the upper-middle price segment achieved **0.07768**. The IDOTRR neighborhood was also
considerably more difficult than several other neighborhoods.

In a real deployment, predictions for rare or poorly represented property segments should
therefore be treated with greater uncertainty and monitored separately.

### Conclusion

The Kaggle result does not invalidate the cross-validation experiment. Instead, the
OOF-to-leaderboard gap provides additional evidence that validation results must be interpreted
together with error analysis, segment performance, and model limitations.

No additional model tuning is performed solely in response to the public leaderboard score.

## Final Conclusions
This project focused not only on achieving a competitive prediction score, but also on building
a methodologically correct and interpretable machine learning workflow for the Ames Housing
dataset.

### 1. Leakage-Free Preprocessing

The most important methodological improvement was moving all learned preprocessing operations
inside the cross-validation pipeline.

Imputation, categorical encoding, and scaling are fitted only on the training portion of each
fold. The validation fold is used only for transformation and prediction.

The Kaggle test dataset is never used to fit imputers, encoders, scalers, or models.

This removes preprocessing data leakage and makes the OOF validation results more trustworthy.

### 2. Structural Outlier Analysis

The initial leakage-free Ridge baseline produced:

- OOF RMSLE: **0.14838**
- Fold standard deviation: **0.04050**

OOF error analysis revealed two extreme observations, Id 524 and Id 1299. Both properties had
exceptionally large living areas and high overall quality, but unusually low sale prices.

Instead of removing observations simply because they produced large prediction errors, a
transparent structural rule was investigated:

**GrLivArea > 4000 and SalePrice < 300000**

This rule identified exactly these two observations.

After applying the structural rule and retraining the complete leakage-free pipeline, Ridge
improved to:

- OOF RMSLE: **0.11549**
- Fold standard deviation: **0.00819**

This demonstrates that the decision was based on inspection and domain-related structure rather
than simply deleting observations that were difficult for the model.

### 3. Model Comparison

The leakage-free models produced the following OOF results:

| Model | OOF RMSLE |
|---|---:|
| Ridge | **0.11549** |
| XGBoost | **0.11537** |
| LightGBM | **0.12361** |

XGBoost was only slightly better than Ridge as an individual model.

However, the OOF error correlation between Ridge and XGBoost was **0.83920**. Their errors were
therefore related but not identical, providing a justified reason to test an ensemble.

### 4. Final Ensemble

A controlled OOF blending experiment showed that the best tested combination was:

- **50% Ridge**
- **50% XGBoost**

The resulting OOF RMSLE was:

**0.11069**

The fold results were:

- Fold 1: **0.11664**
- Fold 2: **0.10544**
- Fold 3: **0.11515**
- Fold 4: **0.11577**
- Fold 5: **0.09937**

The fold standard deviation was **0.00688**, indicating reasonably stable performance across
the five validation folds.

### 5. Why LightGBM Was Excluded

LightGBM achieved a weaker standalone OOF RMSLE of **0.12361**.

Its OOF error correlation with XGBoost was also high at **0.92182**.

A controlled three-model blending experiment tested increasing LightGBM contributions.
The best result occurred with:

**LightGBM weight = 0%**

Therefore, LightGBM was excluded from the final ensemble because it increased model complexity
without improving OOF performance.

### 6. Error Analysis and Model Limitations

The final Ridge/XGBoost ensemble showed very little overall systematic bias:

- Mean log residual: **0.00071**
- Median log residual: **0.00395**
- Mean absolute log error: **0.07482**
- Median absolute log error: **0.05283**

However, model performance was not uniform across the housing market.

Performance by price range was:

| Price Segment | RMSLE |
|---|---:|
| Low | **0.15106** |
| Lower-Middle | **0.09920** |
| Upper-Middle | **0.07768** |
| High | **0.10152** |

The model performs best in the middle of the price distribution and is substantially less
reliable for low-priced properties.

Performance also varied across OverallQual groups and neighborhoods.

For example:

- IDOTRR RMSLE: **0.24541**
- OldTown RMSLE: **0.14567**
- CollgCr RMSLE: **0.06488**
- Gilbert RMSLE: **0.07403**

Results for very small groups should be interpreted cautiously because their error estimates
are based on only a few observations.

These results demonstrate why a single global RMSLE score is not sufficient to describe the
reliability of the model.

### 7. Kaggle Generalization

The final model achieved:

- Leakage-free OOF RMSLE: **0.11069**
- Kaggle Public Score: **0.12808**
- Generalization gap: **0.01739**

The public Kaggle score is worse than the cross-validation estimate, indicating that the OOF
estimate was somewhat optimistic relative to unseen competition data.

Rather than repeatedly tuning the model against the public leaderboard, this difference is
treated as evidence of remaining generalization uncertainty.

Repeated tuning against the public leaderboard would effectively turn it into another
validation set and could lead to leaderboard overfitting.

### Final Takeaway

The main improvement in this project is not simply a lower validation score.

The final workflow establishes a clear relationship between:

**hypothesis → experiment → OOF evaluation → error analysis → decision**

The Ridge/XGBoost ensemble was selected because the OOF evidence supported the combination.
LightGBM was evaluated but rejected because it did not provide sufficient additional value.

The project also identifies important limitations of the final model. Prediction reliability
varies across price ranges, property quality levels, and neighborhoods. In a real-world
deployment, predictions for rare, low-priced, or poorly represented properties should therefore
be treated with greater uncertainty and monitored separately.

The final model should be viewed as a validated predictive system with documented limitations,
rather than only as a Kaggle leaderboard score.